# Notebook 07d — BSISO Supervised-2D Collapse Fix (daily-average data)
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

Fixes the **training collapse** seen in `nb07c` (raw-dot-product InfoNCE, 2-D, no L2 norm) when run on the **daily-average** `X_MJJAS_lee.npy`. Implements the Session-37 plan with a **cheap-fix-first** structure:

1. **Step 0 — diagnose + cheapest fix.** Instrument collapse metrics, reproduce the baseline collapse, check per-channel input variance (snapshot→daily change), and test whether **unit-variance re-standardization of X alone** restores stability. If it does, we may stop here.
2. **Staged OFAT sweep** (only if needed): batch size → temperature → structural anti-collapse (raw / VICReg-reg / L2+amplitude-head) → weight decay → `ReduceLROnPlateau` → early stopping → combine + seeds.

**Collapse metrics** (per epoch, on active-BSISO val embeddings):
- eigenvalue ratio `λ₂/λ₁` of the 2-D embedding covariance (line-collapse < 0.05)
- effective rank `(Σλ)²/Σλ²` ∈ [1,2] (healthy > 1.3)
- angular entropy over 36 bins (healthy → log 36)
- norm mean/std/max (explosion > 100; point-collapse → 0)

**Baselines (BSISO, year-split):** 64-D 67.7% phase / z 3.83 · 2-D L2-norm 33.2% / z 2.59 (1-D-circle ceiling) · random phase 12.5%.

**Pass gate:** eff-rank > 1.3 AND λ₂/λ₁ > 0.2 · max-norm < 100 · phase > 33% (stretch ≥ 62%) · ENSO z ≥ 2.5 · stable val loss.

**Inputs** (`data/processed/`): `X_MJJAS_lee.npy` (N≈6579, 3, 31, 51), `labels_aligned_mjjas_lee.csv`.  
**Runtime:** Step 0 ≈ 2 runs; each later stage 3–4 runs. Run stages incrementally — partial results persist to JSON.

---

## Cell 1 — Setup + Collapse-Metric Instrumentation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import matplotlib.pyplot as plt

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
RESULTS_DIR    = f'{PROJECT_DIR}/results/sup2d_collapse_sweep'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

X_FILE      = 'X_MJJAS_lee.npy'
LABELS_FILE = 'labels_aligned_mjjas_lee.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


def collapse_metrics(Z):
    """Geometry diagnostics for a 2-D embedding matrix Z (n, 2).
    Returns eig_ratio (λ2/λ1), eff_rank ((Σλ)²/Σλ²), ang_entropy (nats),
    and norm mean/std/max."""
    Z = np.asarray(Z, dtype=np.float64)
    norms = np.linalg.norm(Z, axis=1)
    cov = np.cov(Z.T)                       # (2,2)
    ev = np.sort(np.linalg.eigvalsh(cov))[::-1]
    ev = np.clip(ev, 0, None)
    lam1 = ev[0] + 1e-12
    eig_ratio = float(ev[1] / lam1)
    eff_rank = float((ev.sum() ** 2) / (np.sum(ev ** 2) + 1e-12))
    ang = np.arctan2(Z[:, 1], Z[:, 0])
    hist, _ = np.histogram(ang, bins=36, range=(-np.pi, np.pi))
    p = hist / (hist.sum() + 1e-12)
    ang_entropy = float(-(p[p > 0] * np.log(p[p > 0])).sum())   # max = log(36)=3.584
    return {'eig_ratio': eig_ratio, 'eff_rank': eff_rank,
            'ang_entropy': ang_entropy,
            'norm_mean': float(norms.mean()), 'norm_std': float(norms.std()),
            'norm_max': float(norms.max())}


def is_collapsed(m):
    """Pass-gate geometry check."""
    return not (m['eff_rank'] > 1.3 and m['eig_ratio'] > 0.2 and m['norm_max'] < 100)

print('Collapse metrics ready.  Max angular entropy = log(36) =', round(np.log(36), 3))

## Cell 2 — Load Data + Year Split + Phase-ENSO Index + Input-Variance Check

Same split/index logic as nb07c. Also prints per-channel input variance and auto-detects a snapshot backup for the optional A/B (skipped gracefully if absent).

In [ ]:
X_raw  = np.load(f'{PROCESSED_DIR}/{X_FILE}').astype(np.float32)
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])
print(f'X shape: {X_raw.shape}   labels: {len(labels)} rows')
assert X_raw.shape[0] == len(labels), 'X / labels length mismatch'

all_years   = sorted(labels['date'].dt.year.unique())
val_years   = all_years[::5]
train_years = [y for y in all_years if y not in val_years]
year_col    = labels['date'].dt.year
train_idx   = labels.index[year_col.isin(train_years)].values
val_idx     = labels.index[year_col.isin(val_years)].values
print(f'Val years ({len(val_years)}): {val_years}')
print(f'Train: {len(train_idx)}   Val: {len(val_idx)}')

phase_enso_index = defaultdict(list)
for idx in train_idx:
    row = labels.loc[idx]
    if row['bsiso_amplitude'] > 1.0:
        phase_enso_index[(row['bsiso_phase'], row['enso_category'])].append(idx)
active_total = sum(len(v) for v in phase_enso_index.values())
print(f'Active BSISO train days (amp>1): {active_total}/{len(train_idx)}')

# Per-channel input variance (train samples) — the suspected daily-average trigger
ch_names = ['u850', 'v850', 'OLR']
ch_std  = X_raw[train_idx].std(axis=(0, 2, 3))
ch_mean = X_raw[train_idx].mean(axis=(0, 2, 3))
print('\nPer-channel input stats (train):')
for c, mu, sd in zip(ch_names, ch_mean, ch_std):
    print(f'  {c}: mean={mu:+.4f}  std={sd:.4f}')

# Unit-variance re-standardized copy (Stage 0.5 cheap fix)
X_renorm = ((X_raw - ch_mean[None, :, None, None]) / (ch_std[None, :, None, None] + 1e-6)).astype(np.float32)
print('\nX_renorm per-channel std (should be ~1):', X_renorm[train_idx].std(axis=(0, 2, 3)).round(3))

# Optional snapshot A/B
SNAP_CANDIDATES = ['X_MJJAS_lee_snapshot.npy', 'X_MJJAS_lee_snapshot12z.npy',
                   '_snapshot12z_backup/X_MJJAS_lee.npy']
X_snap = None
for cand in SNAP_CANDIDATES:
    p = f'{PROCESSED_DIR}/{cand}'
    if os.path.exists(p):
        X_snap = np.load(p).astype(np.float32)
        print(f'\nSnapshot processed array found: {cand}  -> A/B available')
        break
if X_snap is None:
    print('\nNo snapshot processed array found -> A/B skipped (variance check + renorm test cover the trigger).')

## Cell 3 — PairSampler + Dataset (returns amplitudes for the L2+amp variant)

In [ ]:
class PairSampler:
    def __init__(self, labels_df, index):
        self.labels = labels_df
        self.index  = index
        self.enso_categories = labels_df['enso_category'].unique().tolist()
    def _random_category(self):
        valid = [k for k in self.index if len(self.index[k]) > 0]
        return valid[np.random.randint(len(valid))]
    def sample_positive_pair(self):
        key = self._random_category(); indices = self.index[key]
        if len(indices) < 2:
            return self.sample_easy_negative_pair()
        idx_A, idx_B = np.random.choice(indices, size=2, replace=False)
        year_A = self.labels.loc[idx_A, 'date'].year
        other = [i for i in indices if self.labels.loc[i, 'date'].year != year_A]
        if other: idx_B = np.random.choice(other)
        return idx_A, idx_B
    def sample_hard_negative_pair(self):
        if len(self.enso_categories) < 2:
            return self.sample_easy_negative_pair()
        phase = np.random.choice(range(1, 9))
        eA, eB = np.random.choice(self.enso_categories, size=2, replace=False)
        kA, kB = (phase, eA), (phase, eB)
        if not self.index[kA] or not self.index[kB]:
            return self.sample_positive_pair()
        return np.random.choice(self.index[kA]), np.random.choice(self.index[kB])
    def sample_easy_negative_pair(self):
        pA, pB = np.random.choice(range(1, 9), size=2, replace=False)
        eA = np.random.choice(self.enso_categories); eB = np.random.choice(self.enso_categories)
        kA, kB = (pA, eA), (pB, eB)
        idx_A = (np.random.choice(self.index[kA]) if self.index[kA]
                 else self.labels[self.labels['bsiso_phase'] == pA].sample(1).index[0])
        idx_B = (np.random.choice(self.index[kB]) if self.index[kB]
                 else self.labels[self.labels['bsiso_phase'] == pB].sample(1).index[0])
        return idx_A, idx_B


class PairDataset(Dataset):
    def __init__(self, X_data, labels_df, index, mode, indices):
        self.X = X_data; self.labels = labels_df
        self.sampler = PairSampler(labels_df, index)
        self.mode = mode; self.indices = indices
        self.amp = labels_df['bsiso_amplitude'].values.astype(np.float32)
        if mode == 'val':
            self.val_pairs = self._val_pairs()
    def _val_pairs(self):
        pairs = []; vl = self.labels.loc[self.indices]
        for ph in range(1, 9):
            for en in self.sampler.enso_categories:
                g = vl[(vl['bsiso_phase'] == ph) & (vl['enso_category'] == en)].index.tolist()
                for i in range(len(g)):
                    for j in range(i + 1, len(g)):
                        pairs.append((g[i], g[j]))
        return pairs[:1000]
    def __len__(self):
        return len(self.indices) if self.mode == 'train' else len(self.val_pairs)
    def __getitem__(self, i):
        if self.mode == 'train':
            r = np.random.rand()
            if r < 0.30:   a, b = self.sampler.sample_positive_pair()
            elif r < 0.50: a, b = self.sampler.sample_hard_negative_pair()
            else:          a, b = self.sampler.sample_easy_negative_pair()
        else:
            a, b = self.val_pairs[i]
        fa = torch.from_numpy(self.X[a]).float(); fb = torch.from_numpy(self.X[b]).float()
        return fa, fb, float(self.amp[a]), float(self.amp[b])

print('Dataset / sampler ready.')

## Cell 4 — Encoder + Loss Variants (raw / vicreg / l2_amp)

Encoder returns a raw 2-D embedding `z` and a scalar amplitude prediction `amp` (from the shared 32-D feature). The amplitude head is only used by the `l2_amp` variant.

In [ ]:
class SupEncoder(nn.Module):
    def __init__(self, embedding_dim=2):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 32, 3, padding=1, bias=False); self.bn3 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc       = nn.Linear(32, embedding_dim)
        self.amp_head = nn.Linear(32, 1)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        feat = self.global_pool(x).view(x.size(0), -1)   # (B, 32)
        return self.fc(feat), self.amp_head(feat).squeeze(-1)


def infonce_raw(zA, zB, tau):
    sim = torch.matmul(zA, zB.T) / tau
    tgt = torch.arange(zA.size(0), device=zA.device)
    return F.cross_entropy(sim, tgt)


def vicreg_terms(z, gamma=1.0, eps=1e-4):
    z = z - z.mean(0, keepdim=True)
    std = torch.sqrt(z.var(0) + eps)
    var_loss = torch.mean(F.relu(gamma - std))
    B, D = z.shape
    cov = (z.T @ z) / (B - 1)
    off = cov - torch.diag(torch.diag(cov))
    cov_loss = off.pow(2).sum() / D
    return var_loss, cov_loss


def compute_loss(zA, zB, ampA, ampB, ampA_t, ampB_t, cfg):
    """Return total loss for the configured variant."""
    v = cfg['variant']
    if v == 'raw':
        return infonce_raw(zA, zB, cfg['temperature'])
    if v == 'vicreg':
        nce = infonce_raw(zA, zB, cfg['temperature'])
        vA, cA = vicreg_terms(zA); vB, cB = vicreg_terms(zB)
        return nce + cfg.get('lambda_v', 25.0) * (vA + vB) / 2 + cfg.get('lambda_c', 1.0) * (cA + cB) / 2
    if v == 'l2_amp':
        nce = infonce_raw(F.normalize(zA, dim=1), F.normalize(zB, dim=1), cfg['temperature'])
        amp = F.mse_loss(ampA, ampA_t) + F.mse_loss(ampB, ampB_t)
        return nce + cfg.get('lambda_amp', 1.0) * amp
    raise ValueError(v)


# sanity
_enc = SupEncoder().to(device)
_z, _a = _enc(torch.randn(4, 3, 31, 51).to(device))
print(f'params={sum(p.numel() for p in _enc.parameters()):,}  z={tuple(_z.shape)}  amp={tuple(_a.shape)}')

## Cell 5 — `run(config, X_data)` Harness + Evaluation

Trains one configuration, tracks collapse metrics per epoch on active-BSISO val embeddings, supports cosine **or** `ReduceLROnPlateau` scheduling and early stopping (restore best weights), then evaluates phase probe + ENSO z. Returns a metrics dict + per-epoch history.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

active_mask_all = (labels['bsiso_amplitude'].values > 1.0)
val_active_idx  = np.intersect1d(val_idx, np.where(active_mask_all)[0])


def extract_emb(encoder, X_data, idx):
    encoder.eval(); out = np.zeros((len(idx), 2), np.float32)
    with torch.no_grad():
        for s in range(0, len(idx), 256):
            b = torch.from_numpy(X_data[idx[s:s+256]]).float().to(device)
            z, _ = encoder(b); out[s:s+256] = z.cpu().numpy()
    return out


def enso_z(emb_all):
    phases = range(1, 9); obs = []
    for ph in phases:
        mEN = (labels['bsiso_phase'] == ph) & (labels['enso_category'] == 'El Nino')
        mLN = (labels['bsiso_phase'] == ph) & (labels['enso_category'] == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: continue
        obs.append(np.linalg.norm(emb_all[mEN.values].mean(0) - emb_all[mLN.values].mean(0)))
    rng = np.random.default_rng(42); base = []
    for _ in range(100):
        shuf = labels['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values; t = []
        for ph in phases:
            mph = (labels['bsiso_phase'] == ph).values
            mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
            if mEN.sum() < 3 or mLN.sum() < 3: continue
            t.append(np.linalg.norm(emb_all[mEN].mean(0) - emb_all[mLN].mean(0)))
        if t: base.append(np.mean(t))
    bmu, bsd = np.mean(base), np.std(base)
    return float((np.mean(obs) - bmu) / (bsd + 1e-8))


def run(config, X_data, epochs=None, seed=42, tag='run', verbose=True):
    cfg = dict(config); epochs = epochs or cfg.get('epochs', 50)
    torch.manual_seed(seed); np.random.seed(seed)
    tr = PairDataset(X_data, labels, phase_enso_index, 'train', train_idx)
    va = PairDataset(X_data, labels, phase_enso_index, 'val',   val_idx)
    trl = DataLoader(tr, batch_size=cfg['batch_size'], shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    val = DataLoader(va, batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    enc = SupEncoder().to(device)
    opt = optim.Adam(enc.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    if cfg.get('scheduler', 'cosine') == 'plateau':
        sch = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5, min_lr=1e-6)
    else:
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    hist = {'train_loss': [], 'val_loss': [], 'eig_ratio': [], 'eff_rank': [],
            'ang_entropy': [], 'norm_max': []}
    best_val, best_state, bad = float('inf'), None, 0
    es_pat = cfg.get('es_patience', None)
    for ep in range(epochs):
        enc.train(); tl = 0.0
        for fa, fb, aa, ab in trl:
            fa, fb = fa.to(device), fb.to(device)
            aa, ab = aa.to(device), ab.to(device)
            zA, pA = enc(fa); zB, pB = enc(fb)
            loss = compute_loss(zA, zB, pA, pB, aa, ab, cfg)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0); opt.step()
            tl += loss.item()
        tl /= len(trl)
        enc.eval(); vl = 0.0
        with torch.no_grad():
            for fa, fb, aa, ab in val:
                fa, fb = fa.to(device), fb.to(device); aa, ab = aa.to(device), ab.to(device)
                zA, pA = enc(fa); zB, pB = enc(fb)
                vl += compute_loss(zA, zB, pA, pB, aa, ab, cfg).item()
        vl /= max(len(val), 1)
        sch.step(vl) if cfg.get('scheduler', 'cosine') == 'plateau' else sch.step()
        cm = collapse_metrics(extract_emb(enc, X_data, val_active_idx))
        hist['train_loss'].append(tl); hist['val_loss'].append(vl)
        for k in ['eig_ratio', 'eff_rank', 'ang_entropy', 'norm_max']:
            hist[k].append(cm[k])
        if vl < best_val - 1e-4:
            best_val, best_state, bad = vl, copy.deepcopy(enc.state_dict()), 0
        else:
            bad += 1
        if es_pat and bad >= es_pat:
            if verbose: print(f'  early stop @ ep {ep+1}')
            break
    if es_pat and best_state is not None:
        enc.load_state_dict(best_state)
    emb_all = extract_emb(enc, X_data, np.arange(len(X_data)))
    cm_final = collapse_metrics(emb_all[val_active_idx])
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(emb_all[train_idx], labels.loc[train_idx, 'bsiso_phase'].values)
    phase_val = float(accuracy_score(labels.loc[val_idx, 'bsiso_phase'].values, clf.predict(emb_all[val_idx])))
    z = enso_z(emb_all)
    res = {'tag': tag, 'config': {k: cfg[k] for k in cfg},
           'phase_val': phase_val, 'enso_z': z,
           'collapsed': is_collapsed(cm_final), **{f'final_{k}': v for k, v in cm_final.items()},
           'epochs_run': len(hist['train_loss']), 'best_val_loss': best_val}
    if verbose:
        print(f"[{tag}] phase={phase_val*100:.1f}%  z={z:.2f}  eff_rank={cm_final['eff_rank']:.2f}  "
              f"eig={cm_final['eig_ratio']:.3f}  nmax={cm_final['norm_max']:.1f}  "
              f"collapsed={res['collapsed']}")
    return res, hist, enc

print('run() ready.')

## Cell 6 — Step 0: Baseline Collapse + Unit-Variance Renorm Cheap-Fix Test

Run the **current nb07c config** on the raw daily-average X (should collapse), then on the unit-variance-renormalized X. If renorm alone clears the pass gate, we may adopt it and skip the full sweep (per the agreed open question).

In [ ]:
BASE_CFG = {'variant': 'raw', 'batch_size': 64, 'temperature': 0.5,
            'lr': 1e-3, 'weight_decay': 1e-4, 'scheduler': 'cosine', 'epochs': 50}

print('=== Step 0a: baseline (raw daily-average X, nb07c config) ===')
res_base, hist_base, _ = run(BASE_CFG, X_raw, tag='step0_baseline')

print('\n=== Step 0b: cheap fix (unit-variance renormalized X, same config) ===')
res_renorm, hist_renorm, _ = run(BASE_CFG, X_renorm, tag='step0_renorm')

SWEEP_RESULTS = [res_base, res_renorm]
with open(f'{RESULTS_DIR}/sweep_results.json', 'w') as f:
    json.dump(SWEEP_RESULTS, f, indent=2)

# plot collapse-metric trajectories side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for h, lab in [(hist_base, 'baseline (raw X)'), (hist_renorm, 'renorm X')]:
    axes[0].plot(h['eff_rank'], label=lab)
    axes[1].plot(h['eig_ratio'], label=lab)
    axes[2].plot(h['norm_max'], label=lab)
axes[0].axhline(1.3, color='r', ls='--', lw=1); axes[0].set_title('effective rank (>1.3 ok)')
axes[1].axhline(0.2, color='r', ls='--', lw=1); axes[1].set_title('eig ratio λ2/λ1 (>0.2 ok)')
axes[2].axhline(100, color='r', ls='--', lw=1); axes[2].set_title('max norm (<100 ok)')
for ax in axes: ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/step0_trajectories.png', dpi=130, bbox_inches='tight'); plt.show()

cheap_fix_works = (not res_renorm['collapsed']) and res_renorm['phase_val'] > 0.33 and res_renorm['enso_z'] >= 2.5
print('\n' + '=' * 60)
print(f"Baseline collapsed: {res_base['collapsed']}  (phase {res_base['phase_val']*100:.1f}%, z {res_base['enso_z']:.2f})")
print(f"Renorm   collapsed: {res_renorm['collapsed']}  (phase {res_renorm['phase_val']*100:.1f}%, z {res_renorm['enso_z']:.2f})")
print('=' * 60)
if cheap_fix_works:
    print('CHEAP FIX WORKS -> adopt unit-variance renorm; full sweep optional.')
else:
    print('Cheap fix insufficient -> run the staged sweep (Cells 7+). Use X_renorm as the base X.')

## Cell 7 — Staged OFAT Sweep (run only if the cheap fix was insufficient)

Each stage appends to `SWEEP_RESULTS` and re-saves the JSON, so you can run stages one at a time. `BEST` carries the winning settings forward. Base X is `X_renorm` (renorm is essentially free and removes one confound). Edit `STAGE` to pick which block to execute, or run them top-to-bottom.

> Runtime note: each `run(...)` ≈ the cost of one nb07c training. Lower `epochs` (e.g. 40) to speed up the sweep; lock the winner at full epochs.

In [ ]:
X_BASE = X_renorm   # renorm base for all sweep runs
SWEEP_EPOCHS = 40   # shorter for sweeping; final recipe re-run at 50+

def add(res):
    SWEEP_RESULTS.append(res)
    with open(f'{RESULTS_DIR}/sweep_results.json', 'w') as f:
        json.dump(SWEEP_RESULTS, f, indent=2)

BEST = dict(BASE_CFG)   # will be updated stage by stage

# ---- Stage 1: batch size ----
for bs in [64, 128, 256]:
    cfg = dict(BEST, batch_size=bs, epochs=SWEEP_EPOCHS)
    r, _, _ = run(cfg, X_BASE, epochs=SWEEP_EPOCHS, tag=f's1_bs{bs}'); add(r)
BEST['batch_size'] = max([r for r in SWEEP_RESULTS if r['tag'].startswith('s1_')],
                         key=lambda r: (not r['collapsed'], r['phase_val']))['config']['batch_size']
print('Stage 1 winner batch_size =', BEST['batch_size'])

# ---- Stage 2: temperature ----
for tau in [0.07, 0.1, 0.2, 0.5]:
    cfg = dict(BEST, temperature=tau, epochs=SWEEP_EPOCHS)
    r, _, _ = run(cfg, X_BASE, epochs=SWEEP_EPOCHS, tag=f's2_tau{tau}'); add(r)
BEST['temperature'] = max([r for r in SWEEP_RESULTS if r['tag'].startswith('s2_')],
                          key=lambda r: (not r['collapsed'], r['phase_val']))['config']['temperature']
print('Stage 2 winner temperature =', BEST['temperature'])

# ---- Stage 3: structural anti-collapse variant ----
for var in ['raw', 'vicreg', 'l2_amp']:
    cfg = dict(BEST, variant=var, epochs=SWEEP_EPOCHS)
    r, _, _ = run(cfg, X_BASE, epochs=SWEEP_EPOCHS, tag=f's3_{var}'); add(r)
BEST['variant'] = max([r for r in SWEEP_RESULTS if r['tag'].startswith('s3_')],
                      key=lambda r: (not r['collapsed'], r['phase_val'], r['enso_z']))['config']['variant']
print('Stage 3 winner variant =', BEST['variant'])

# ---- Stage 4: weight decay ----
for wd in [1e-4, 5e-4, 1e-3]:
    cfg = dict(BEST, weight_decay=wd, epochs=SWEEP_EPOCHS)
    r, _, _ = run(cfg, X_BASE, epochs=SWEEP_EPOCHS, tag=f's4_wd{wd}'); add(r)
BEST['weight_decay'] = max([r for r in SWEEP_RESULTS if r['tag'].startswith('s4_')],
                           key=lambda r: (not r['collapsed'], r['phase_val']))['config']['weight_decay']
print('Stage 4 winner weight_decay =', BEST['weight_decay'])

# ---- Stage 5: LR schedule (plateau vs cosine) x base lr ----
for sched in ['cosine', 'plateau']:
    for lr in [1e-3, 3e-4]:
        cfg = dict(BEST, scheduler=sched, lr=lr, epochs=SWEEP_EPOCHS)
        r, _, _ = run(cfg, X_BASE, epochs=SWEEP_EPOCHS, tag=f's5_{sched}_lr{lr}'); add(r)
w5 = max([r for r in SWEEP_RESULTS if r['tag'].startswith('s5_')],
         key=lambda r: (not r['collapsed'], r['phase_val']))
BEST['scheduler'] = w5['config']['scheduler']; BEST['lr'] = w5['config']['lr']
print('Stage 5 winner scheduler/lr =', BEST['scheduler'], BEST['lr'])

print('\nCarried-forward BEST so far:', BEST)

## Cell 8 — Stage 6+7: Early Stopping + Final Recipe over 3 Seeds

In [ ]:
FINAL_CFG = dict(BEST, scheduler='plateau', es_patience=10, epochs=80)
print('Final recipe:', FINAL_CFG)

seed_res = []
for sd in [42, 1, 7]:
    r, h, enc = run(FINAL_CFG, X_BASE, epochs=80, seed=sd, tag=f'final_seed{sd}'); add(r); seed_res.append(r)
    if sd == 42:
        torch.save(enc.state_dict(), f'{CHECKPOINT_DIR}/encoder_sup2d_fixed_final.pth')

pv = np.array([r['phase_val'] for r in seed_res]); zz = np.array([r['enso_z'] for r in seed_res])
er = np.array([r['final_eff_rank'] for r in seed_res])
print('\n=== FINAL RECIPE (3 seeds) ===')
print(f'phase val: {pv.mean()*100:.1f}% ± {pv.std()*100:.1f}%   (L2 ceiling 33%, 64D 67.7%)')
print(f'ENSO z:    {zz.mean():.2f} ± {zz.std():.2f}   (L2 2.59, 64D 3.83)')
print(f'eff rank:  {er.mean():.2f} ± {er.std():.2f}   (collapse if <1.3)')

with open(f'{CHECKPOINT_DIR}/sup2d_fixed_config.json', 'w') as f:
    json.dump({'config': FINAL_CFG, 'renorm': True,
               'phase_val_mean': float(pv.mean()), 'phase_val_std': float(pv.std()),
               'enso_z_mean': float(zz.mean()), 'enso_z_std': float(zz.std()),
               'eff_rank_mean': float(er.mean())}, f, indent=2)
print('Saved final config + checkpoint.')

## Cell 9 — Results Table + Decision

In [ ]:
df = pd.DataFrame([{
    'tag': r['tag'], 'variant': r['config'].get('variant'),
    'bs': r['config'].get('batch_size'), 'tau': r['config'].get('temperature'),
    'wd': r['config'].get('weight_decay'), 'sched': r['config'].get('scheduler'),
    'phase%': round(r['phase_val'] * 100, 1), 'z': round(r['enso_z'], 2),
    'eff_rank': round(r['final_eff_rank'], 2), 'eig': round(r['final_eig_ratio'], 3),
    'nmax': round(r['final_norm_max'], 1), 'collapsed': r['collapsed'],
} for r in SWEEP_RESULTS])
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/sweep_table.csv', index=False)

ok = df[(~df['collapsed']) & (df['phase%'] > 33)]
print('\n' + '=' * 60)
if len(ok):
    best = ok.sort_values(['phase%', 'z'], ascending=False).iloc[0]
    print(f"BEST NON-COLLAPSED: {best['tag']}  phase={best['phase%']}%  z={best['z']}  eff_rank={best['eff_rank']}")
    if best['phase%'] >= 62 and best['z'] >= 3.0:
        print('VERDICT: GREENLIGHT — BSISO genuinely 2-D; use this recipe for nb08 SSL temporal.')
    else:
        print('VERDICT: PARTIAL — beats the 33% 1-D-circle ceiling but below 64-D; discuss vs dim sweep.')
else:
    print('VERDICT: still collapsing across the sweep — escalate (dim sweep nb07e / revisit inputs).')
print('=' * 60)
print('Saved: sweep_table.csv, sweep_results.json, step0_trajectories.png')

---
## Done!

**Send back:** `step0_trajectories.png`, the Step-0 printout (did renorm alone fix it?), and `sweep_table.csv` if you ran the full sweep.

- If **Step 0 cheap fix works** → adopt unit-variance renorm in nb07c (and nb03 going forward), skip the sweep.
- If not → the staged sweep picks a non-collapsed recipe; `encoder_sup2d_fixed_final.pth` + `sup2d_fixed_config.json` are the locked outputs for nb08.

Snapshot data: keep the raw `_snapshot12z_backup/` only; daily-average is canonical going forward.

---
*DDCS Project | jh9141@nyu.edu*